# regime 2 / Compare Results — LoRA vs QLoRA vs two-factor vs three-factor
Loads `results/{lora,qlora,two_factor,three_factor}.json` from Drive (run those first; missing ones are skipped).

## 1. Setup + Load Results

In [ ]:
import os, json, math, torch
import matplotlib.pyplot as plt
USE_DRIVE, DRIVE_SUBDIR = True, 'Section8_regime2'
if USE_DRIVE:
    try:
        from google.colab import drive; drive.mount('/content/drive'); STORE = os.path.join('/content/drive/MyDrive', DRIVE_SUBDIR)
    except Exception as e:
        print('Drive mount failed:', e); STORE = os.path.join('/content', DRIVE_SUBDIR)
else:
    STORE = os.path.join('.', DRIVE_SUBDIR)
RESULTS_DIR = os.path.join(STORE, 'results')
METHODS = ['lora', 'qlora', 'two_factor', 'three_factor']
LABELS  = {'lora':'LoRA (backprop)', 'qlora':'QLoRA (4-bit)', 'two_factor':'Two-factor Hebbian', 'three_factor':'Three-factor (this work)'}
COLORS  = {'lora':'#1F3864', 'qlora':'#2E7D32', 'two_factor':'#B8860B', 'three_factor':'#C62828'}
R = {}
for m in METHODS:
    p = os.path.join(RESULTS_DIR, f'{m}.json')
    if os.path.exists(p):
        R[m] = json.load(open(p)); s = R[m]['summary']; md = R[m]['meta']
        print(f'loaded {m:14} {md["total_steps"]:>8,} steps  {md["wall_clock_sec"]/3600:5.2f}h  final acc {s["final_acc"]:.3f}  best loss {s["best_loss"]:.4f}')
    else:
        print(f'MISSING {p} (run the {m} notebook first)')

## 2. Configuration

In [ ]:
for m in R:
    md=R[m]['meta']; print('='*60); print(LABELS[m]); print('  P_train=%d epochs=%.3f'%(md.get('P_trainable',0),md['epochs'])); print('  config:', json.dumps(md['config']))

## 3. Compare Loss Curves

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
for m in R:
    c = R[m]['curve']; hrs = [t/3600 for t in c['t_sec']]
    ax[0].plot(hrs, c['test_loss'], 'o-', color=COLORS[m], label=LABELS[m])
    ax[1].plot(hrs, c['test_acc'],  'o-', color=COLORS[m], label=LABELS[m])
ax[0].set_xlabel('wall-clock hours'); ax[0].set_ylabel('test loss'); ax[0].set_title('CIFAR-10 test loss vs time'); ax[0].legend()
ax[1].set_xlabel('wall-clock hours'); ax[1].set_ylabel('test accuracy'); ax[1].set_title('CIFAR-10 test accuracy vs time'); ax[1].legend()
plt.tight_layout(); plt.show()

## 4. Compare Cost & Memory

In [ ]:
def fwd_equiv(m):
    st = R[m]['meta']['total_steps']
    if m == 'three_factor': return st * 2 * R[m]['meta']['config'].get('M', 0)  # loop probes
    if m in ('lora', 'qlora'): return st * 3                                    # fwd + ~2x bwd
    return st * 2                                                               # two-factor
print(f'{"method":26}{"P_train":>10}{"steps":>9}{"fwd-equiv":>13}{"wall h":>8}{"peak MB":>9}')
print('-'*75)
for m in R:
    md = R[m]['meta']
    print(f'{LABELS[m]:26}{md.get("P_trainable",0):>10,}{md["total_steps"]:>9,}{fwd_equiv(m):>13,}'
          f'{md["wall_clock_sec"]/3600:>8.2f}{md.get("peak_mem_mb", float("nan")):>9.1f}')

## 5. Summary table

In [ ]:
print(f'{"Experiment":26}{"Epochs":>9}{"First loss":>12}{"Final loss":>12}{"Best loss":>11}{"Reduc %":>9}{"Test acc":>10}')
print('-'*89)
for m in R:
    md, s = R[m]['meta'], R[m]['summary']
    print(f'{LABELS[m]:26}{md["epochs"]:>9.3f}{s["initial_loss"]:>12.4f}{s["final_loss"]:>12.4f}'
          f'{s["best_loss"]:>11.4f}{s["reduction_pct"]:>9.1f}{s["final_acc"]:>10.3f}')
print('\n(Equal wall-clock: LoRA/QLoRA get thousands of steps; the forward-only three-factor rule gets a few '
      'hundred because each probe is a full ViT-Base forward -- the honest cost of backprop-free big-model adaptation.)')